In [5]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\Sudhe\AppData\Local\Temp\ipykernel_14012\2860160806.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Sudhe\anaconda3\envs\sudheer_agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
#print(api_key)

In [22]:
BASE_DIR = Path.cwd()
print(BASE_DIR)
DATA_DIR = BASE_DIR

#DATA_DIR = Path(
#    r"D:\MAHA\AIPro\AgenticAI\myAgenticAI_7am_May26"
#    r"\Data"
#)

preferred_pdf = DATA_DIR / "Sudheer_Real_Estate_HR_Policies.pdf"

if preferred_pdf.exists():
    PDF_PATH = preferred_pdf
else:
    # Automatically find the PDF if its filename is slightly different
    available_pdfs = sorted(DATA_DIR.glob("*.pdf"))

    if len(available_pdfs) == 1:
        PDF_PATH = available_pdfs[0]
    elif len(available_pdfs) == 0:
        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )
    else:
        raise RuntimeError(
            "Multiple PDF files were found. Please set PDF_PATH manually.\n"
            + "\n".join(str(path) for path in available_pdfs)
        )

print("PDF found:")
print(PDF_PATH)

c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel
PDF found:
c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf


In [23]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 10


In [24]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estate Company - HR Policies', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Sudheer Real Estate
Company
Comprehensive Human Resources Policy Manual
Document ID: HR-POL-2026-V1.4
Effective Date: January 1, 2026
Last Revised: August 15, 2026
Prepared By: Department of Human Resources 
Proprietary and Confidential
For Internal Use Only 
1


In [25]:
def identify_sudheer_hr_section(paper_page: int) -> str:
    """
    Identify the major section of the Sudheer Real Estate HR Policies
    using its printed PDF page number.
    """

    if paper_page == 1:
        return "table_of_contents_and_introduction"

    if 2 <= paper_page <= 3:
        return "employment_categories_and_code_of_conduct"

    if 4 <= paper_page <= 5:
        return "working_hours_leaves_and_compensation"

    if 6 <= paper_page <= 7:
        return "performance_travel_and_workplace_safety"

    if 8 <= paper_page <= 9:
        return "it_assets_separation_and_acknowledgment"

    if paper_page == 10:
        return "signature_block"

    return "unknown"

In [26]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "document_title": "Sudheer Real Estate Company HR Policies", #[cite: 1]
            "organization": "Sudheer Real Estate Company", #[cite: 1]
            "year": 2026, #[cite: 1]
            "document_type": "hr_policy_manual", #[cite: 1]
            "paper_page": paper_page,
            "section": identify_sudheer_hr_section(paper_page),
            "access_level": "internal_confidential", #[cite: 1]
        }
    )

In [27]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estate Company - HR Policies', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'document_title': 'Sudheer Real Estate Company HR Policies', 'organization': 'Sudheer Real Estate Company', 'year': 2026, 'document_type': 'hr_policy_manual', 'paper_page': 1, 'section': 'table_of_contents_and_introduction', 'access_level': 'internal_confidential'}
{'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estate Company - HR Policies', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'document_title': 'Sudheer Real Estate Company HR Policies', 'organization': 'Sudheer Real Estate Company', 'year': 2026, 'document_type

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}


In [28]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 10
Total chunks: 22


In [29]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [30]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Sudheer Real Estate
Company
Comprehensive Human Resources Policy Manual
Document ID: HR-POL-2026-V1.4
Effective Date: January 1, 2026
Last Revised: August 15, 2026
Prepared By: Department of Human Resources 
Proprietary and Confidential
For Internal Use Only 
1

Chunk metadata:
{'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estate Company - HR Policies', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'document_title': 'Sudheer Real Estate Company HR Policies', 'organization': 'Sudheer Real Estate Company', 'year': 2026, 'document_type': 'hr_policy_manual', 'paper_page': 1, 'section': 'table_of_contents_and_introduction', 'access_level': 'internal_confidential', 'start_index': 0, 'chunk_id': 'llama2-page-1-chunk-0'}


In [31]:

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [32]:
# Generate the embedding vector for your query
test_vector = embeddings.embed_query(
    "What are the working hours in Sudheer Real Estate?"
)

# Print the results to verify
print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [-0.02886962890625, 0.0291900634765625, 0.08380126953125, -0.00045108795166015625, -0.0277557373046875, 0.044036865234375, 0.033966064453125, 0.032623291015625, 0.036865234375, -0.007755279541015625]


In [33]:
PERSIST_DIRECTORY = DATA_DIR / "sudheer_RE_retriever"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [34]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="sudheer_RE_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 22
Persisted at: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\sudheer_RE_retriever


In [35]:
# Same directory used during creation
PERSIST_DIRECTORY = DATA_DIR / "sudheer_RE_retriever"

In [36]:
PERSIST_DIRECTORY

WindowsPath('c:/Sudheer/Agentic_AI/workspace/python/AI_Basics/Basics_2/retrivel/sudheer_RE_retriever')

In [37]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="sudheer_RE_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\sudheer_RE_retriever


In [74]:
def display_documents(
    documents,
    max_characters: int = 200
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [28]:
# vector_store.similarity_search()

In [75]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

# Retrival 

In [76]:
query = "What is remote work policy in Sudheer Real Estate?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 1
SECTION: table_of_contents_and_introduction
CHUNK ID: llama2-page-1-chunk-0
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
Sudheer Real Estate
Company
Comprehensive Human Resources Policy Manual
Document ID: HR-POL-2026-V1.4
Effective Date: January 1, 2026
Last Revised: August 15, 2026
Prepared By: Department of Human Res

RANK: 2
PAPER PAGE: 3
SECTION: employment_categories_and_code_of_conduct
CHUNK ID: llama2-page-3-chunk-3
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
1.2 Mission and Vision
Mission: To provide unparalleled real estate services by maintaining the highest standards of integrity,
market knowledge, and client satisfaction.
Vision: To be the mo

In [77]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [78]:
query = " Maternity and Paternity Leave policy?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 6
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-6-chunk-11
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
5.2 Maternity and Paternity Leave
Female employees who have worked for a minimum of 80 days in the preceding 12 months are entitled
to 26 weeks of paid Maternity Leave. Male employees are entitled to 

RANK: 2
PAPER PAGE: 9
SECTION: it_assets_separation_and_acknowledgment
CHUNK ID: llama2-page-9-chunk-19
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
notice  period  is  30  days  for  junior  staff,  60  days  for  mid-management,  and  90  days  for  senior
management. The company reserves the right to accept payment in lieu of the 

In [79]:
query = "Appraisal Cycle in Sudheer Real Estate?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [80]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
7 performance_travel_and_workplace_safety llama2-page-7-chunk-13
3 employment_categories_and_code_of_conduct llama2-page-3-chunk-3
4 working_hours_leaves_and_compensation llama2-page-4-chunk-6
2 employment_categories_and_code_of_conduct llama2-page-2-chunk-1

MMR results:
7 performance_travel_and_workplace_safety llama2-page-7-chunk-13
10 signature_block llama2-page-10-chunk-20
3 employment_categories_and_code_of_conduct llama2-page-3-chunk-5
9 it_assets_separation_and_acknowledgment llama2-page-9-chunk-19


In [81]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.40,
    }
)

In [82]:
query = "Appraisal Cycle in Sudheer Real Estate?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-13
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
6.4 Health Insurance
All full-time employees and their immediate dependents (spouse and up to two children) are covered
under  the  company's  Group  Mediclaim  Policy,  providing  coverage  up  to  I

RANK: 2
PAPER PAGE: 3
SECTION: employment_categories_and_code_of_conduct
CHUNK ID: llama2-page-3-chunk-3
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
1.2 Mission and Vision
Mission: To provide unparalleled real estate services by maintaining the highest standards of integrity,
market knowledge, and client satisfaction.
Vision: To be 

In [37]:
# threshold_retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 10,
#         "score_threshold": 0.35,
#     }


In [83]:
query = "What are  Business Ethics in Sudheer REal Estate Company?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.6683
Paper page: 3
Section: employment_categories_and_code_of_conduct
1.2 Mission and Vision
Mission: To provide unparalleled real estate services by maintaining the highest standards of integrity,
market knowledge, and client satisfaction.
Vision: To be the most trusted and innovative real estate enterprise in the region, redefining urban living
and commercial spaces.
1.3 Core Values
Integrity: Upholding honesty and ethical standards in every property transaction.
Excellence: Delivering outstanding quality in our construction projects and advisory services.
Colla
Rank: 2
Relevance score: 0.6377
Paper page: 4
Section: working_hours_leaves_and_compensation
2.3 Probationary Period
All new full-time employees must undergo a probationary period of six (6) months. This period allows
both the company and the employee to evaluate the working relationship. During this time, employment
may be terminated by either party with a notice period of fifteen (15) days. Upon s

In [84]:
metric_query = "What is Probationary Period in Sudheer Real Estate Company?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [85]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)


In [86]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [87]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,4,working_hours_leaves_and_compensation,llama2-page-4-chunk-6,0.758814,0.694569,0.758898,2.3 Probationary Period All new full-time empl...
1,3,employment_categories_and_code_of_conduct,llama2-page-3-chunk-3,0.597229,0.897830,0.597643,1.2 Mission and Vision Mission: To provide unp...
2,2,employment_categories_and_code_of_conduct,llama2-page-2-chunk-1,0.582917,0.913659,0.583340,Table of Contents 1. Introduction & Company Ov...
3,7,performance_travel_and_workplace_safety,llama2-page-7-chunk-13,0.548278,0.950479,0.548256,6.4 Health Insurance All full-time employees a...
4,1,table_of_contents_and_introduction,llama2-page-1-chunk-0,0.545685,0.953313,0.545790,Sudheer Real Estate Company Comprehensive Huma...
5,8,it_assets_separation_and_acknowledgment,llama2-page-8-chunk-16,0.539055,0.960222,0.539135,"including hard hats, high-visibility vests..."


In [88]:
metric_query

'What is Probationary Period in Sudheer Real Estate Company?'

In [89]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,4,working_hours_leaves_and_compensation,0.758814,2.3 Probationary Period All new full-time empl...
1,3,employment_categories_and_code_of_conduct,0.597229,1.2 Mission and Vision Mission: To provide unp...
2,2,employment_categories_and_code_of_conduct,0.582917,Table of Contents 1. Introduction & Company Ov...
3,7,performance_travel_and_workplace_safety,0.548278,6.4 Health Insurance All full-time employees a...
4,1,table_of_contents_and_introduction,0.545685,Sudheer Real Estate Company Comprehensive Huma...
5,8,it_assets_separation_and_acknowledgment,0.539055,"including hard hats, high-visibility vests..."


In [90]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,4,working_hours_leaves_and_compensation,0.694569,2.3 Probationary Period All new full-time empl...
1,3,employment_categories_and_code_of_conduct,0.897830,1.2 Mission and Vision Mission: To provide unp...
2,2,employment_categories_and_code_of_conduct,0.913659,Table of Contents 1. Introduction & Company Ov...
3,7,performance_travel_and_workplace_safety,0.950479,6.4 Health Insurance All full-time employees a...
4,1,table_of_contents_and_introduction,0.953313,Sudheer Real Estate Company Comprehensive Huma...
5,8,it_assets_separation_and_acknowledgment,0.960222,"including hard hats, high-visibility vests..."


In [91]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,4,working_hours_leaves_and_compensation,0.758898,2.3 Probationary Period All new full-time empl...
1,3,employment_categories_and_code_of_conduct,0.597643,1.2 Mission and Vision Mission: To provide unp...
2,2,employment_categories_and_code_of_conduct,0.583340,Table of Contents 1. Introduction & Company Ov...
3,7,performance_travel_and_workplace_safety,0.548256,6.4 Health Insurance All full-time employees a...
4,1,table_of_contents_and_introduction,0.545790,Sudheer Real Estate Company Comprehensive Huma...
5,8,it_assets_separation_and_acknowledgment,0.539135,"including hard hats, high-visibility vests..."


In [92]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.75881382 0.5972293  0.58291658 0.54827751 0.54568512 0.53905518]

Dot product after normalization:
[0.75881382 0.5972293  0.58291658 0.54827751 0.54568512 0.53905518]

Are they approximately equal? True


In [97]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "it_assets_separation_and_acknowledgment"
        },
    }
)

In [98]:
query = "What is the notice period for resignation?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 9
SECTION: it_assets_separation_and_acknowledgment
CHUNK ID: llama2-page-9-chunk-19
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
notice  period  is  30  days  for  junior  staff,  60  days  for  mid-management,  and  90  days  for  senior
management. The company reserves the right to accept payment in lieu of the notice period 

RANK: 2
PAPER PAGE: 9
SECTION: it_assets_separation_and_acknowledgment
CHUNK ID: llama2-page-9-chunk-18
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
11.2 Data Privacy
Customer databases, architectural drawings, and pricing matrices are highly confidential. Downloading
bulk client data to personal external drives is disabled by defaul

In [100]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "it_assets_separation_and_acknowledgment"

print("All returned documents are from the it_assets_separation_and_acknowledgment section.")

All returned documents are from the it_assets_separation_and_acknowledgment section.


In [110]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        # Updated to the section containing Travel Policies
                        "$eq": "performance_travel_and_workplace_safety"
                    }
                },
                {
                    "year": {
                        "$eq": 2026
                    }
                },
                {
                    "organization": {
                        # Updated to match your document's metadata
                        "$eq": "Sudheer Real Estate Company"
                    }
                },
            ]
        },
    }
)

In [111]:
query = "Domestic and International Travel Of Sudheer Real Estate?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-13
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
6.4 Health Insurance
All full-time employees and their immediate dependents (spouse and up to two children) are covered
under  the  company's  Group  Mediclaim  Policy,  providing  coverage  up  to  I

RANK: 2
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-14
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Em

Prefilter

In [112]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "performance_travel_and_workplace_safety"
            }
        },
        {
            "year": {
                "$eq": 2026
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "Domestic and International Travel Of Sudheer Real Estate?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-13
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
6.4 Health Insurance
All full-time employees and their immediate dependents (spouse and up to two children) are covered
under  the  company's  Group  Mediclaim  Policy,  providing  coverage  up  to  I

RANK: 2
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-14
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Em

Post-Filtering

In [113]:
unfiltered_candidates = vector_store.similarity_search(
    query="Domestic and International Travel Of Sudheer Real Estate?",
    k=15,
)

In [114]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "performance_travel_and_workplace_safety"
    and document.metadata.get("year") == 2026
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-13
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
6.4 Health Insurance
All full-time employees and their immediate dependents (spouse and up to two children) are covered
under  the  company's  Group  Mediclaim  Policy,  providing  coverage  up  to  I

RANK: 2
PAPER PAGE: 7
SECTION: performance_travel_and_workplace_safety
CHUNK ID: llama2-page-7-chunk-14
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\retrivel\Sudheer_Real_Estate_HR_Policies.pdf
------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Em

In [115]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 3


05-Aug-2026

In [116]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [117]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [118]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [119]:
from langchain_community.retrievers import BM25Retriever

In [120]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [121]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000273FACB3AD0>)

In [122]:
# Final number of results
bm25_retriever.k = 4

In [123]:
sparse_query = "performance_travel_and_workplace_safety"

In [124]:
sparse_documents = bm25_retriever.invoke(sparse_query)

In [126]:
display_documents(
    sparse_documents,
    title="performance_travel_and_workplace_safety: BM25 Results",
)


performance_travel_and_workplace_safety: BM25 Results

RANK: 1
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-21
----------------------------------------------------------------------------------------------------
13. Acknowledgment of Receipt
I acknowledge that I have received a copy of the Sudheer Real Estate Company HR Policies Manual. I
understand that it is my responsibility to read and comply with the policies contained in this manual and
any revisions made to it. I understand that this manual does not constitute a contract of employment.
Employee Signature Date
Employee Name (Printed) Employee ID
Sudheer Real Estate Company | Confidential HR Policy Manual | Version 1.4 | Page output generated for RAG Knowledge Base.
10

RANK: 2
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-20
----------------------------------------------------------------------------------------------------
12.2 Handover Process
A comprehensive handover of ongoi

In [127]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [128]:
dense_query = (
    "performance_travel_and_workplace_safety?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
----------------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Employees consistently falling below expectations will be placed on a 60-day Performance Improvement
Plan.  The  PIP  outlines  specific,  measurable  goals.  Failure  to  meet  these  objectives  may  lead  to
reassignment or termination of employment.
8. Travel, Site Visits & Expense Reimbursement
8.1 Local Conveyance for Site Visits
Sales executives and site engineers are required to travel extensively within 

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
------------------------------------------------

In [129]:
comparison_query = (
    "performance_travel_and_workplace_safety?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-21
----------------------------------------------------------------------------------------------------
13. Acknowledgment of Receipt
I acknowledge that I have received a copy of the Sudheer Real Estate Company HR Policies Manual. I
understand that it is my responsibility to read and comply with the policies contained in this manual and
any revisions made to it. I understand that this manual does not constitute a contract of employment.
Employee Signature Date
Employee Name (Printed) Employee ID
Sudheer Real Estate Company | Confidential HR Policy Manual | Version 1.4 | Page output generated for RAG Knowledge Base.
10

RANK: 2
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-20
----------------------------------------------------------------------------------------------------
12.2 Handover Process
A comprehensive handover of ongoing projects, client portfolios, and compa

In [130]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 10 llama2-page-10-chunk-21
2 10 llama2-page-10-chunk-20
3 2 llama2-page-2-chunk-1
4 2 llama2-page-2-chunk-2

DENSE RESULTS
1 7 llama2-page-7-chunk-14
2 8 llama2-page-8-chunk-15
3 8 llama2-page-8-chunk-16
4 9 llama2-page-9-chunk-17


In [131]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [132]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [133]:
hybrid_query = (
    "performance_travel_and_workplace_safety?"
)

In [134]:
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [135]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 2
Section: employment_categories_and_code_of_conduct
Chunk ID: llama2-page-2-chunk-2
----------------------------------------------------------------------------------------------------
outlines the policies, procedures, and expectations that govern our working relationship. We believe in
fostering an environment of transparency, growth, and mutual respect.
2

RANK: 2
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-21
----------------------------------------------------------------------------------------------------
13. Acknowledgment of Receipt
I acknowledge that I have received a copy of the Sudheer Real Estate Company HR Policies Manual. I
understand that it is my responsibility to read and comply with the policies contained in this manual and
any revisions made to it. I understand that this manual does not constitute a contract of employment.
Employee Signature Date
Employee Name (Printed) Employee I

                         ┌── BM25 Retriever ─────┐
User query ──────────────┤                       ├── Weighted RRF
                         └── Dense Retriever ────┘
                                                    ↓
                                            Combined ranking

In [136]:
test_query = (
    "performance_travel_and_workplace_safety?"
)

In [137]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [138]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-21
----------------------------------------------------------------------------------------------------
13. Acknowledgment of Receipt
I acknowledge that I have received a copy of the Sudheer Real Estate Company HR Policies Manual. I
understand that it is my responsibility to read and comply with the policies contained in this manual and
any revisions made to it. I understand that this manual does not constitute a contract of employment.
Employee Signature Date
Employee Name (Printed) Employee ID
Sudheer Real Estate Company | Confidential HR Policy Manual | Version 1.4 | Page output generated for RAG Knowledge Base.
10

RANK: 2
Paper page: 10
Section: signature_block
Chunk ID: llama2-page-10-chunk-20
----------------------------------------------------------------------------------------------------
12.2 Handover Process
A comprehensive handover of ongoing projects, client portfolios, an

In [139]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: gpt-4.1-mini


In [140]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [141]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

In [142]:
chat_history = """
User: performance_travel_and_workplace_safety?
Assistant: It first underwent supervised fine-tuning.
"""

In [143]:
original_query = "performance_travel_and_workplace_safety?"

In [144]:
rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

In [145]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

Original query:
performance_travel_and_workplace_safety?

Rewritten query:
Performance metrics and best practices for travel and workplace safety


In [146]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

In [147]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)


Documents Retrieved Using the Rewritten Query

RANK: 1
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
----------------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Employees consistently falling below expectations will be placed on a 60-day Performance Improvement
Plan.  The  PIP  outlines  specific,  measurable  goals.  Failure  to  meet  these  objectives  may  lead  to
reassignment or termination of employment.
8. Travel, Site Visits & Expense Reimbursement
8.1 Local Conveyance for Site Visits
Sales executives and site engineers are required to travel extensively within 

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
-----------------------------------------

In [148]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [149]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

In [150]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [151]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

In [152]:
original_query = (
    "performance_travel_and_workplace_safety?"
)


In [153]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

In [154]:
expanded_output

ExpandedQueryOutput(queries=['workplace safety and travel performance', 'travel safety and employee performance', 'performance metrics for travel and workplace safety', 'occupational safety and travel efficiency'])

In [155]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]


In [156]:
all_expanded_queries

['performance_travel_and_workplace_safety?',
 'workplace safety and travel performance',
 'travel safety and employee performance',
 'performance metrics for travel and workplace safety',
 'occupational safety and travel efficiency']

In [157]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

Generated search queries:

1. performance_travel_and_workplace_safety?
2. workplace safety and travel performance
3. travel safety and employee performance
4. performance metrics for travel and workplace safety
5. occupational safety and travel efficiency


In [158]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)


Query Expansion: Combined Unique Documents

RANK: 1
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
----------------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Employees consistently falling below expectations will be placed on a 60-day Performance Improvement
Plan.  The  PIP  outlines  specific,  measurable  goals.  Failure  to  meet  these  objectives  may  lead  to
reassignment or termination of employment.
8. Travel, Site Visits & Expense Reimbursement
8.1 Local Conveyance for Site Visits
Sales executives and site engineers are required to travel extensively within 

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
--------------------------------------------

In [159]:
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

In [160]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

In [161]:
multi_query_documents = multi_query_retriever.invoke(
    "performance_travel_and_workplace_safety?"
)


In [162]:
display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)


Built-in MultiQueryRetriever Results

RANK: 1
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
----------------------------------------------------------------------------------------------------
8.2 Domestic and International Travel
For inter-city travel (e.g., visiting outstation project sites or investor meetings), travel and accommodation
must be pre-approved by the Department Head. Employees are expected to book economy class flights
and adhere to standard corporate hotel limits based on their grade.
8.3 Client Entertainment
Reasonable expenses incurred for entertaining prospective HNI (High Net-worth Individual) clients or
corporate investors are reimbursable. Prior approval is required for expenses exceeding INR 5,000.
Alcohol expenses are strictly capped and monitored.
9. Workp

RANK: 2
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
--------------------------------------------------

In [163]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [164]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [165]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [166]:
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

In [167]:
complex_query = """
performance_travel_and_workplace_safety?.
"""

In [168]:
decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)


In [169]:
sub_queries = decomposed_output.sub_queries

In [170]:
print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

Original complex query:

performance_travel_and_workplace_safety?.


Generated sub-queries:
1. Performance metrics related to travel safety in the workplace
2. Workplace safety standards and regulations for employees who travel
3. Impact of travel on employee performance and workplace safety
4. Best practices for ensuring safety during work-related travel


In [171]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [172]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )


Sub-query: Performance metrics related to travel safety in the workplace

RANK: 1
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
----------------------------------------------------------------------------------------------------
8.2 Domestic and International Travel
For inter-city travel (e.g., visiting outstation project sites or investor meetings), travel and accommodation
must be pre-approved by the Department Head. Employees are expected to book economy class flights
and adhere to standard corporate hotel limits based on their grade.
8.3 Client Entertainment
Reasonable expenses incurred for entertaining prospective HNI (High Net-worth Individual) clients or
corporate investors are reimbursable. Prior approval is required for expenses exceeding INR 5,000.
Alcohol expenses are strictly capped and monitored.
9. Workp

RANK: 2
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
--------------

In [173]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)


Combined Evidence from All Sub-Queries

RANK: 1
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
----------------------------------------------------------------------------------------------------
8.2 Domestic and International Travel
For inter-city travel (e.g., visiting outstation project sites or investor meetings), travel and accommodation
must be pre-approved by the Department Head. Employees are expected to book economy class flights
and adhere to standard corporate hotel limits based on their grade.
8.3 Client Entertainment
Reasonable expenses incurred for entertaining prospective HNI (High Net-worth Individual) clients or
corporate investors are reimbursable. Prior approval is required for expenses exceeding INR 5,000.
Alcohol expenses are strictly capped and monitored.
9. Workp

RANK: 2
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
------------------------------------------------

Complex query
      ↓
Atomic sub-queries
      ↓
Retrieve for every sub-query
      ↓
Merge evidence
      ↓
Remove duplicates

## HyDE (Hypothetical Document Embeddings)

HyDE asks an LLM to write a plausible passage that could answer the query. The hypothetical passage is embedded and used for dense retrieval instead of embedding the short user query directly. The generated text is only a search aid; the final answer must still be grounded in retrieved source documents.

In [174]:
from typing import Any

from langchain_core.retrievers import BaseRetriever


class HyDERetriever(BaseRetriever):
    """Generate a hypothetical passage and retrieve real documents with it."""

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager,
    ):
        hypothetical_document = self.hypothesis_chain.invoke(
            {"query": query}
        ).strip()

        self.last_hypothetical_document = hypothetical_document

        return self.vector_retriever.invoke(
            hypothetical_document,
            config={
                "callbacks": run_manager.get_child(),
            },
        )

In [175]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.
""",
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

hyde_dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=hyde_dense_retriever,
)

In [176]:
hyde_query = (
    "performance_travel_and_workplace_safety?"
)

hyde_documents = hyde_retriever.invoke(hyde_query)

print("Original query:")
print(hyde_query)

print("\nHypothetical document used for retrieval:")
print(hyde_retriever.last_hypothetical_document)

display_documents(
    hyde_documents,
    title="HyDE Retrieval Results",
    max_documents=5,
)

Original query:
performance_travel_and_workplace_safety?

Hypothetical document used for retrieval:
Performance in travel and workplace safety is critically influenced by the implementation of comprehensive risk management protocols, adherence to regulatory standards such as OSHA (Occupational Safety and Health Administration) guidelines, and the integration of advanced safety technologies. Effective safety performance metrics include incident rate reduction, near-miss reporting frequency, and compliance audit scores. In travel contexts, performance is enhanced through rigorous vehicle maintenance schedules, driver training programs emphasizing defensive driving techniques, and real-time monitoring systems utilizing GPS and telematics to mitigate risks. Workplace safety performance is further optimized by ergonomic assessments, hazard identification processes, and the deployment of personal protective equipment (PPE) tailored to specific occupational hazards. The synergy of these eleme

User query
     ↓
LLM-generated hypothetical document
     ↓
Embed the hypothetical document
     ↓
Dense vector search
     ↓
Real source documents

## HyDE with LangChain's built-in helper

`HypotheticalDocumentEmbedder` is an embeddings wrapper, not a complete retriever. For a query, it generates a hypothetical document with the LLM and embeds that generated text. Chroma then uses the resulting vector to search the collection of real PDF chunks.

In [177]:
try:
    # LangChain v1: legacy chains live in langchain-classic.
    from langchain_classic.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )
except ImportError:
    # Compatibility fallback for older LangChain versions.
    from langchain.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )

In [178]:
built_in_hyde_embeddings = (
    HypotheticalDocumentEmbedder.from_llm(
        llm=llm,
        base_embeddings=embeddings,
        prompt_key="web_search",
    )
)

# Reuse the existing collection. Real PDF chunks remain embedded with
# `embeddings`; HyDE is used only to transform query embeddings.
built_in_hyde_vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=built_in_hyde_embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = (
    built_in_hyde_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )
)

In [179]:
built_in_hyde_query = (
    "performance_travel_and_workplace_safety?"
)

built_in_hyde_documents = (
    built_in_hyde_retriever.invoke(
        built_in_hyde_query
    )
)

display_documents(
    built_in_hyde_documents,
    title="Built-in LangChain HyDE Results",
    max_documents=5,
)


Built-in LangChain HyDE Results
No documents were returned.


> **Note:** Do not use `built_in_hyde_embeddings` when initially indexing the PDF chunks. Index real documents with the normal `OpenAIEmbeddings` object, then use the HyDE wrapper only when loading the collection for retrieval.

In [180]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

Vector search
     ↓
Top 20 candidates

In [181]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

In [182]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [183]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [184]:
reranking_query = (
    "performance_travel_and_workplace_safety?"
)

In [185]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

In [186]:
initial_candidates

[Document(id='2aa2fb14-a210-46ed-aff0-530b58782678', metadata={'title': 'Sudheer Real Estate Company - HR Policies', 'document_type': 'hr_policy_manual', 'total_pages': 10, 'chunk_id': 'llama2-page-7-chunk-14', 'section': 'performance_travel_and_workplace_safety', 'creationdate': '', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'page': 6, 'organization': 'Sudheer Real Estate Company', 'access_level': 'internal_confidential', 'producer': 'WeasyPrint 62.3', 'paper_page': 7, 'creator': 'PyPDF', 'start_index': 852, 'page_label': '7', 'document_title': 'Sudheer Real Estate Company HR Policies', 'year': 2026}, page_content='and material optimization.\nCorporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to\ncompany policies.\n7.3 Performance Improvement Plan (PIP)\nEmployees consistently falling below expectations will be placed on a 60-day Performance Improvement\nPlan.

In [187]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)


Before Reranking: Initial Vector Candidates

RANK: 1
Paper page: 7
Section: performance_travel_and_workplace_safety
Chunk ID: llama2-page-7-chunk-14
----------------------------------------------------------------------------------------------------
and material optimization.
Corporate Staff: Process efficiency, inter-departmental support, project delivery, and adherence to
company policies.
7.3 Performance Improvement Plan (PIP)
Employees consistently falling below expectations will be placed on a 60-day Performance Improvement
Plan.  The  PIP  outlines  specific,  measurable  goals.  Failure  to  meet  these  objectives  may  lead  to
reassignment or termination of employment.
8. Travel, Site Visits & Expense Reimbursement
8.1 Local Conveyance for Site Visits
Sales executives and site engineers are required to travel extensively within 

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
-------------------------------------------

In [188]:
reranked_documents = reranking_retriever.invoke(reranking_query)

In [189]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)


After Reranking: Final Top Documents

RANK: 1
Paper page: 2
Section: employment_categories_and_code_of_conduct
Chunk ID: llama2-page-2-chunk-1
----------------------------------------------------------------------------------------------------
Table of Contents
1. Introduction & Company Overview
2. Employment Categories & Hiring Policy
3. Code of Conduct & Business Ethics
4. Working Hours, Attendance & Remote Work
5. Leave Policy & Public Holidays
6. Compensation, Commissions & Benefits
7. Performance Management & Appraisals
8. Travel, Site Visits & Expense Reimbursement
9. Workplace Safety, Health & Security
10. Anti-Harassment & POSH Policy
11. IT Assets, Data Security & Social Media
12. Separation, Resignation & Termination
13. Acknowledgment of Receipt
1. Introduction & Company Overview
1.1 Welcome Message
Welcome to Sudheer Rea

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
-------------------------------------------------

User query
     ↓
Dense Retriever
     ↓
Top 20 candidates
     ↓
Cross-Encoder Reranker
     ↓
Query-document relevance evaluation
     ↓
Final top 5 documents

In [190]:
bm25_retriever.k = 15

In [191]:
dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

In [192]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [193]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [194]:
query = (
    "performance_travel_and_workplace_safety?"
)

In [195]:
final_documents = hybrid_reranking_retriever.invoke(query)

In [196]:
final_documents

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sudheer Real Estate Company - HR Policies', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\retrivel\\Sudheer_Real_Estate_HR_Policies.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'document_title': 'Sudheer Real Estate Company HR Policies', 'organization': 'Sudheer Real Estate Company', 'year': 2026, 'document_type': 'hr_policy_manual', 'paper_page': 2, 'section': 'employment_categories_and_code_of_conduct', 'access_level': 'internal_confidential', 'start_index': 0, 'chunk_id': 'llama2-page-2-chunk-1'}, page_content='Table of Contents\n1. Introduction & Company Overview\n2. Employment Categories & Hiring Policy\n3. Code of Conduct & Business Ethics\n4. Working Hours, Attendance & Remote Work\n5. Leave Policy & Public Holidays\n6. Compensation, Commissions & Benefits\n7. Performance Management & Appraisals\n8. Travel, Site Visits & Expense Reimburseme

In [197]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)


Hybrid Retrieval + Cross-Encoder Reranking

RANK: 1
Paper page: 2
Section: employment_categories_and_code_of_conduct
Chunk ID: llama2-page-2-chunk-1
----------------------------------------------------------------------------------------------------
Table of Contents
1. Introduction & Company Overview
2. Employment Categories & Hiring Policy
3. Code of Conduct & Business Ethics
4. Working Hours, Attendance & Remote Work
5. Leave Policy & Public Holidays
6. Compensation, Commissions & Benefits
7. Performance Management & Appraisals
8. Travel, Site Visits & Expense Reimbursement
9. Workplace Safety, Health & Security
10. Anti-Harassment & POSH Policy
11. IT Assets, Data Security & Social Media
12. Separation, Resignation & Termination
13. Acknowledgment of Receipt
1. Introduction & Company Overview
1.1 Welcome Message
Welcome to Sudheer Rea

RANK: 2
Paper page: 8
Section: it_assets_separation_and_acknowledgment
Chunk ID: llama2-page-8-chunk-15
-------------------------------------------

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

Sparse Retrieval
    = BM25Retriever

Dense Retrieval
    = VectorStoreRetriever

Hybrid Retrieval
    = BM25 + Dense + EnsembleRetriever + Weighted RRF

Query Rewriting
    = Convert a contextual query into a standalone query

Query Expansion
    = Generate related query variations and merge their results

Query Decomposition
    = Split one complex query into atomic searchable queries

HyDE
    = Generate a hypothetical answer passage and use its embedding for dense retrieval

Reranking
    = Retrieve broad candidates and reorder them using a cross-encoder

1. BM25 sparse retrieval
2. Dense vector retrieval
3. Hybrid retrieval
4. Query rewriting
5. Query expansion
6. Query decomposition
7. HyDE retrieval
8. Dense retrieval + reranking
9. Hybrid retrieval + reranking

HOME_WORK
weighted fusion
resiporcal rank fusion
multi query retriever
Multi hop retriever
parenet document retriever
sentence window retriever
contextual compression

CUSTOM_RETRIEVER
Langchain 
langchain provide one baseretriever class ontop of it you can create custom retriever with your own logic(when you are usingh langchain/langgraph)

In [164]:
i will take one more hour reteriever
will discuss about the prompting

SyntaxError: invalid syntax (2074422024.py, line 1)

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate

# llm = ChatOpenAI(
#     model="gpt-4.1-mini",
#     temperature=0
# )

# hyde_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# Generate a short hypothetical document that could answer
# the user's question.

# Do not mention that the document is hypothetical.
# Write it in the style of a factual knowledge-base passage.
# """
#         ),
#         (
#             "human",
#             "{query}"
#         )
#     ]
# )

# query = "How does Llama 2 improve safety?"

# response = llm.invoke(
#     hyde_prompt.format_messages(
#         query=query
#     )
# )

# hypothetical_document = response.content

# print("Original Query:")
# print(query)

# print("\nHypothetical Document:")
# print(hypothetical_document)


# hyde_vector = base_embeddings.embed_query(
#     hypothetical_document
# )

# hyde_documents = vector_store.similarity_search_by_vector(
#     hyde_vector,
#     k=4
# )

# for i, document in enumerate(hyde_documents, start=1):
#     print("=" * 80)
#     print(f"RESULT {i}")
#     print(document.page_content[:1000])

multimodal RAG builder
full flede project

MEGA ASSISGNMENT on Sunday